## Завдання 1:
Для кожної з адміністративних одиниць України завантажити (urllib) тестові структуровані файли, що містять значення VHI-індексу. При зберіганні файлу, до його імені потрібно додати дату та час завантаження. Передбачити повторні запуски скрипту, реалізувати механізм запобігання повторного довантаження та колізії даних

In [1]:
import os
import urllib.request
from datetime import datetime

def download_vhi_data(province_id):
    if not os.path.exists('data'):
        os.makedirs('data')
        
    existing_files = [f for f in os.listdir('data') if f.startswith(f"vhi_id_{province_id}_")]
    if existing_files:
        print(f"Дані для області {province_id} вже існують.")
        return

    url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1=1981&year2=2024&type=Mean"
    vhi_url = urllib.request.urlopen(url)
    
    date_time = datetime.now().strftime("%Y%m%d%H%M%S")
    filename = f"data/vhi_id_{province_id}_{date_time}.csv"
    
    with open(filename, 'wb') as f:
        f.write(vhi_url.read())
    print(f"VHI для області {province_id} завантажено.")

for i in range(1, 28):
    download_vhi_data(i)


Дані для області 1 вже існують.
Дані для області 2 вже існують.
Дані для області 3 вже існують.
VHI для області 4 завантажено.
VHI для області 5 завантажено.
VHI для області 6 завантажено.
VHI для області 7 завантажено.
VHI для області 8 завантажено.
VHI для області 9 завантажено.
VHI для області 10 завантажено.
VHI для області 11 завантажено.
VHI для області 12 завантажено.
VHI для області 13 завантажено.
VHI для області 14 завантажено.
VHI для області 15 завантажено.
VHI для області 16 завантажено.
VHI для області 17 завантажено.
VHI для області 18 завантажено.
VHI для області 19 завантажено.
VHI для області 20 завантажено.
VHI для області 21 завантажено.
VHI для області 22 завантажено.
VHI для області 23 завантажено.
VHI для області 24 завантажено.
VHI для області 25 завантажено.
VHI для області 26 завантажено.
VHI для області 27 завантажено.


## Завдання 2 & 3:
2. Зчитати завантажені текстові файли у pandas dataframe. Здійснити data cleaning: прибрати зайві стовпці, заповнити пропуски, видалити зайвий текст тощо. Додати стовпчики з назвою та індексом області
3. Реалізувати процедуру зміни індексів: в завантажених з NOAA даних області індексуються за англійською абеткою (Province 1 - Cherkasy), потрібно замінити індекси так, щоб області індексувалася за українською абеткою (1 область - Вінницька). 


In [3]:
import os
import pandas as pd

def create_and_clean_dataframe(directory_path):
    reindex_map = {
        1: 22, 2: 24, 3: 23, 4: 25, 5: 3, 6: 4, 7: 8, 8: 19, 9: 20, 10: 5,
        11: 9, 12: 26, 13: 10, 14: 11, 15: 12, 16: 13, 17: 14, 18: 15, 19: 16,
        20: 27, 21: 17, 22: 18, 23: 7, 24: 1, 25: 2, 26: 6, 27: 21
    }

    provinces = {
        1: "Вінницька", 2: "Волинська", 3: "Дніпропетровська", 4: "Донецька",
        5: "Житомирська", 6: "Закарпатська", 7: "Запорізька", 8: "Івано-Франківська",
        9: "Київська", 10: "Кіровоградська", 11: "Луганська", 12: "Львівська",
        13: "Миколаївська", 14: "Одеська", 15: "Полтавська", 16: "Рівненська",
        17: "Сумська", 18: "Тернопільська", 19: "Харківська", 20: "Херсонська",
        21: "Хмельницька", 22: "Черкаська", 23: "Чернівецька", 24: "Чернігівська",
        25: "Республіка Крим", 26: "Київ", 27: "Севастополь"
    }

    frames = []

    for filename in os.listdir(directory_path):
        if filename.endswith(".csv"):
            file_path = os.path.join(directory_path, filename)
            
            # ПЕРЕВІРКА: Якщо файл порожній - ігноруємо його
            if os.stat(file_path).st_size == 0:
                print(f"Пропущено порожній файл: {filename}")
                continue

            try:
                old_id = int(filename.split('_')[2])
                new_id = reindex_map.get(old_id)

                # Читаємо файл
                df = pd.read_csv(file_path, index_col=False, header=1)
                
                # Якщо після зчитування виявилося, що в датафреймі немає даних
                if df.empty:
                    continue

                df.columns = df.columns.str.strip().str.lower()
                df = df.rename(columns={col: 'vhi' for col in df.columns if 'vhi' in col})
                
                df['year'] = df['year'].astype(str).str.replace('<tt><pre>', '', regex=False)
                df = df[df['year'] != '']
                df['year'] = pd.to_numeric(df['year'], errors='coerce')
                df['vhi'] = pd.to_numeric(df['vhi'], errors='coerce')

                df['province_id'] = new_id
                df['province_name'] = provinces.get(new_id, "Невідомо")

                frames.append(df.dropna(subset=['year', 'vhi']))
            
            except Exception as e:
                # Якщо файл все одно видає помилку - виводимо її назву і йдемо далі
                print(f"Помилка при читанні файлу {filename}: {e}")
                continue

    return pd.concat(frames, ignore_index=True)

all_data = create_and_clean_dataframe('data')
all_data.sample(10)

Пропущено порожній файл: vhi_id_3_20260331214310.csv


,year,week,smn,smt,vci,tci,vhi,province_id,province_name
33614,1983.0,23.0,0.404,294.30,37.85,65.28,51.57,1,Вінницька
42743,1986.0,52.0,0.045,255.60,15.73,65.51,40.62,24,Чернігівська
694,1995.0,19.0,0.330,291.62,53.23,51.42,52.32,5,Житомирська
8435,2015.0,12.0,0.130,284.75,50.44,16.84,33.64,10,Кіровоградська
3371,2003.0,44.0,0.174,275.61,54.74,49.26,52.00,9,Київська
21022,1999.0,15.0,0.280,285.71,79.88,38.36,59.12,16,Рівненська
47900,2000.0,9.0,0.035,259.48,18.63,72.44,45.54,3,Дніпропетровська
48590,2013.0,23.0,0.402,300.83,68.69,43.11,55.90,3,Дніпропетровська
39124,2003.0,21.0,0.259,304.34,20.41,11.43,15.92,6,Закарпатська
38259,1986.0,40.0,0.111,291.57,9.41,48.77,29.09,6,Закарпатська


## Завдання 4: 
Реалізувати процедуру зміни індексів: в завантажених з NOAA даних області індексуються за англійською абеткою (Province 1 - Cherkasy), потрібно замінити індекси так, щоб області індексувалася за українською абеткою (1 область - Вінницька). 

In [ ]:
def print_vhi_extremes(df, province_id, start_year, end_year):
    subset = df[(df['province'] == province_id) & 
                (df['year'] >= start_year) & 
                (df['year'] <= end_year)]
    
    print(f"Аналіз VHI для області №{province_id} за період {start_year} - {end_year} рр:")
    
    if not subset.empty:
        print(f"Максимальний показник VHI: {subset['vhi'].max()}")
        print(f"Мінімальний показник VHI:  {subset['vhi'].min()}")
    else:
        print("Помилка: Дані за вказаний період відсутні.")

print_vhi_extremes(all_data, 10, 2000, 2020)

## Завдання 5: Реалізація допоміжних функцій:

1. Отримання ряду VHI для вказаної області за конкретний рік.

2. Отримання даних VHI для кількох вибраних областей за вказаний діапазон років.

In [6]:
def get_vhi_for_year(df, province_id, year):
    return df[(df['province_id'] == province_id) & (df['year'] == year)]

def get_vhi_range(df, province_ids, start_year, end_year):
    return df[(df['province_id'].isin(province_ids)) & 
              (df['year'] >= start_year) & 
              (df['year'] <= end_year)]


print("Результат для однієї області за рік (Вінницька - №1):")
display(get_vhi_for_year(all_data, 1, 2010).sample(5))

print("\nРезультат для діапазону областей та років:")
display(get_vhi_range(all_data, [1, 12, 5], 2000, 2015).sample(10))

Результат для однієї області за рік (Вінницька - №1):


,year,week,smn,smt,vci,tci,vhi,province_id,province_name
35026,2010.0,31.0,0.443,299.45,73.40,20.17,46.79,1,Вінницька
35028,2010.0,33.0,0.403,298.52,60.06,21.21,40.64,1,Вінницька
35027,2010.0,32.0,0.424,299.23,67.21,18.56,42.89,1,Вінницька
35036,2010.0,41.0,0.250,284.58,52.76,52.06,52.41,1,Вінницька
35031,2010.0,36.0,0.338,293.35,49.61,48.03,48.82,1,Вінницька



Результат для діапазону областей та років:


,year,week,smn,smt,vci,tci,vhi,province_id,province_name
35205,2014.0,2.0,0.063,257.38,53.25,47.66,50.45,1,Вінницька
34885,2007.0,46.0,0.179,272.85,82.92,38.32,60.62,1,Вінницька
1645,2013.0,34.0,0.415,293.89,65.94,61.82,63.88,5,Житомирська
12238,2002.0,19.0,0.433,290.81,81.85,47.10,64.47,12,Львівська
12407,2005.0,32.0,0.469,294.01,72.17,44.58,58.37,12,Львівська
12250,2002.0,31.0,0.433,294.02,48.40,47.77,48.08,12,Львівська
1288,2006.0,41.0,0.326,287.64,95.83,7.94,51.88,5,Житомирська
34529,2001.0,2.0,0.058,263.92,49.41,29.56,39.49,1,Вінницька
1525,2011.0,18.0,0.342,293.96,72.54,16.77,44.65,5,Житомирська
34913,2008.0,22.0,0.481,294.57,88.21,58.62,73.41,1,Вінницька
